In [19]:
import time
from google.cloud import storage
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery

The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


In [20]:
project_id = "auspicious-lead-382321"
bucket_name = "movie_data_lpez"
folder_name = "initial_load"
dataset_name = "movie_raw"
region = "us"

storage_client = storage.Client(project=project_id)
bq_client = bigquery.Client(project=project_id)

# Field Decomposition

In this notebook, we decompose the fields from the raw tables that contain more than one property in their value. Specifically, we have two description fields, one in `github_theater_location` and another in `tmdb_person_info` that need to be split up into their individual components.


In [ ]:
%%bigquery
select * from movie_raw.github_theater_location limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_person_info limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_person_info limit 5

# github_theater_location

Split up the location field from the raw `github_theater_location` table and create a staging table from the results:

In [ ]:
%%bigquery
select location, location_array[0] as longtitude, location_array[1] as latitude
from
(select location, split(location, ', ') as location_array
from movie_raw.github_theater_location
limit 5)

In [3]:
%%bigquery
CREATE OR REPLACE TABLE movie_stg.Theater as
SELECT * EXCEPT (location, location_array, data_source, load_time), location_array[0] as longtitude, location_array[1] as latitude, 'github' as data_source, load_time
from
    (select *, split(location, ', ') as location_array
    from movie_raw.github_theater_location)

Query is running:   0%|          |

""


Verify that the raw and staging tables both have the same record counts (i.e. we don't want to lose any records from raw to staging):

In [5]:
%%bigquery
select (select count(*) from movie_raw.github_theater_location) as raw_count,
(select count(*) from movie_stg.github_theater_location) as staging_count

Query is running:   0%|          |

Downloading:   0%|          |

,raw_count,staging_count
0,3319,3319


# tmdb_movie

In [6]:
%%bigquery
CREATE OR REPLACE TABLE movie_stg.tmdb_movie as
SELECT * 
from
    (select * EXCEPT (overview), REGEXP_EXTRACT(overview, r'^Title: .* \(\d{4}\)\. (.*)') as overview
    from movie_raw.tmdb_movie)

Query is running:   0%|          |

""


In [21]:
%%bigquery
select * from movie_stg.tmdb_movie limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,record_date,movie_id,title,original_language,overview,popularity,release_date,genre_ids,vote_average,vote_count,data_source,load_time
0,2024-02-08 00:00:00+00:00,1009921,Reclaim,zh,Title: Reclaim (2022). She is a good woman liv...,23.260,7/29/2022,[],2.0,1,tmdb,2024-02-24 04:04:01.835961+00:00
1,2024-02-16 00:00:00+00:00,1009921,Reclaim,zh,Title: Reclaim (2022). She is a good woman liv...,19.541,7/29/2022,[],2.0,1,tmdb,2024-02-24 04:04:01.835961+00:00
2,2024-02-23 00:00:00+00:00,1009921,Reclaim,zh,Title: Reclaim (2022). She is a good woman liv...,16.199,7/29/2022,[],2.0,1,tmdb,2024-02-24 04:04:01.835961+00:00
3,2024-02-16 00:00:00+00:00,1229831,XXL,sv,Title: XXL (2024). Siblings Enzo and Magda hav...,1.676,1/27/2024,[],0.0,0,tmdb,2024-02-24 04:04:01.835961+00:00
4,2024-02-23 00:00:00+00:00,1229831,XXL,sv,Title: XXL (2024). Siblings Enzo and Magda hav...,1.196,1/27/2024,[],0.0,0,tmdb,2024-02-24 04:04:01.835961+00:00


# tmdb_person

In [12]:
%%bigquery
create or replace table movie_stg.Person as
  select * EXCEPT(load_time),
    'tmdb' as data_source, load_time from movie_raw.tmdb_person_info

Query is running:   0%|          |

""


In [13]:
%%bigquery
select * from movie_stg.Person limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,person_id,name,also_known_as,biography,birthday,deathday,gender,homepage,place_of_birth,imdb_id,known_for_department,popularity,data_source,load_time
0,148182,Thelma Witmer,[],In 1944 she was hired by Walt Disney Studios a...,6/18/1901,7/12/1996,0,None,"Nebraska, USA",nm0936818,Art,1.226,tmdb,2024-01-31 00:14:21.495095+00:00
1,1174725,Gianni Polidori,[],None,11/7/1923,8/19/1992,0,None,"Rome, Lazio, Italy",nm0689115,Art,0.635,tmdb,2024-01-31 00:14:21.495095+00:00
2,1623325,Sydney Bettex,[],None,2/5/1922,None,0,None,"Chertsey, Surrey, England, UK",nm0079327,Art,1.246,tmdb,2024-01-31 00:14:21.495095+00:00
3,10183,Bert Davey,[],None,10/22/1921,None,0,None,"London, England, UK",nm0202700,Art,1.422,tmdb,2024-01-31 00:14:21.495095+00:00
4,9153,Gemma Jackson,[],None,1/1/1951,None,1,None,"Guildford, Surrey, England, UK",nm0413541,Art,0.600,tmdb,2024-01-31 00:14:21.495095+00:00


Split up the `birthday` from the `person_info` table. Original string format: dd/mm/yyyy OR yyyy-mm-dd

Split up the `deathday` from the `person_info` table. Original string format: dd/mm/yyyy OR yyyy-mm-dd

In [14]:
import json, datetime
from google.cloud import bigquery

project_id = "auspicious-lead-382321"
raw_dataset_name = "movie_raw"
raw_table_name = "tmdb_person_info"

stg_dataset_name = "movie_stg"
stg_table_name = "Person"
region = "us"

tmdb_person = []
target_table_id = "{}.{}.{}".format(project_id, stg_dataset_name, stg_table_name)

def serialize_datetime(obj):
    if isinstance(obj, datetime.datetime):
        return obj.isoformat()
    raise TypeError("Type not serializable")

def split_dates(date):
    day = None
    month = None
    year = None 
    if not date:
        pass    
    elif len(date.split("-")) > 1:
        day = int(date.split("-")[2].strip())
        month = int(date.split("-")[1].strip())
        year = int(date.split("-")[0].strip())
    elif len(date.split("/")) > 1:
        day = int(date.split("/")[0].strip())
        month = int(date.split("/")[1].strip())
        year = int(date.split("/")[2].strip())

    return day, month, year


def remove_none_values(record):
    filtered_record = {}
    for field in record.keys():
        if record[field] != None:
            filtered_record[field] = record[field]
    return filtered_record

schema = [
    bigquery.SchemaField("person_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("also_known_as", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("biography", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("birth_day", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("birth_month", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("birth_year", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("death_day", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("death_month", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("death_year", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("gender", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("homepage", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("place_of_birth", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("imdb_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("known_for_department", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("popularity", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED")
]

storage_client = storage.Client(project=project_id)
bq_client = bigquery.Client(project=project_id)
sql = "select * from {}.{}".format(raw_dataset_name, raw_table_name)
query_job = bq_client.query(sql)

# birthday and deathday strings have format: dd/mm/yyyy OR yyyy-mm-dd
for row in query_job:
    person_id = row["person_id"]
    name = row["name"]
    also_known_as = row["also_known_as"]
    biography = row["biography"]
    gender = row["gender"]
    homepage = row["homepage"]
    place_of_birth = row["place_of_birth"]
    imdb_id = row["imdb_id"]
    known_for_department = row["known_for_department"]
    popularity = row["popularity"]
    birthday = row["birthday"]
    deathday = row["deathday"]
    load_time = json.dumps(row["load_time"], default=serialize_datetime).replace('"', '')

    birth_day, birth_month, birth_year = split_dates(birthday)
    death_day, death_month, death_year = split_dates(deathday)

    record = {
        "person_id": person_id,
        "name": name,
        "also_known_as": also_known_as,
        "biography": biography,
        "birth_day": birth_day,
        "birth_month": birth_month,
        "birth_year": birth_year,
        "death_day": death_day,
        "death_month": death_month,
        "death_year": death_year,
        "gender": gender,
        "homepage": homepage,
        "place_of_birth": place_of_birth,
        "imdb_id": imdb_id,
        "known_for_department": known_for_department,
        "popularity": popularity,
        "data_source": "tmdb",
        "load_time": load_time
    }
    filtered_record = remove_none_values(record)
    tmdb_person.append(filtered_record)

# load records into staging table
job_config = bigquery.LoadJobConfig(schema=schema, source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON, write_disposition='WRITE_TRUNCATE')
table_ref = bigquery.table.TableReference.from_string(target_table_id)


try:
    load_job = bq_client.load_table_from_json(tmdb_person, table_ref, job_config=job_config)
    destination_table = bq_client.get_table(target_table_id)
    load_job.result()
    print('Inserted into', stg_table_name, ':', len(tmdb_person), 'records')

    if load_job.errors:
        print('job errors:', job.errors)

except Exception as e:
    print("Error inserting into BQ: {}".format(e))


Inserted into Person : 374906 records


Verify that we ended up with the same record count in the staging table as in the raw table:

In [15]:
%%bigquery
select (select count(*) from movie_raw.tmdb_person_info) as raw_count,
  (select count(*) from movie_stg.Person) as intermediate_stg_count

Query is running:   0%|          |

Downloading:   0%|          |

,raw_count,intermediate_stg_count
0,374906,374906


In [16]:
%%bigquery
select * from movie_stg.Person limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,person_id,name,also_known_as,biography,birth_day,birth_month,birth_year,death_day,death_month,death_year,gender,homepage,place_of_birth,imdb_id,known_for_department,popularity,data_source,load_time
0,4504601,Pam Stephens,[],None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,None,None,None,Art,0.0,tmdb,2024-01-31 00:14:21.495095+00:00
1,4504606,Jon Rudo,[],None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,None,None,None,Art,0.0,tmdb,2024-01-31 00:14:21.495095+00:00
2,4504161,Nicholas Williams,[],None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,None,None,None,Art,0.0,tmdb,2024-01-31 00:14:21.495095+00:00
3,4030373,Kent Adams,[],None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,None,None,None,Art,0.0,tmdb,2024-01-31 00:14:21.495095+00:00
4,1090144,Julie Ann Horan,[],None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,None,None,None,Art,2.0,tmdb,2024-01-31 00:14:21.495095+00:00


# Primary Key

BigQuery does not enforce primary keys, so the following command is for understanding the intent of the `airline_id` field. We will still need to check that it conforms to a PK.

In [3]:
%%bigquery
alter table movie_stg.Theater
  add primary key (id) not enforced

Query is running:   0%|          |

""


In [4]:
%%bigquery
select id, count(*) duplicate_records
from movie_stg.Theater
group by id
having count(*) > 1
order by count(*) desc

Query is running:   0%|          |

Downloading: |          |

,id,duplicate_records


In [17]:
%%bigquery
alter table movie_stg.Person
  add primary key (person_id) not enforced

Query is running:   0%|          |

""


In [18]:
%%bigquery
select person_id, count(*) duplicate_records
from  movie_stg.Person
group by person_id
having count(*) > 1
order by count(*) desc

Query is running:   0%|          |

Downloading: |          |

,person_id,duplicate_records
